# 07 — QLoRA Fine-Tuning

Fine-tune **Qwen3-VL-8B-Instruct** on our schema.org training dataset using QLoRA.

**Run this notebook on RunPod** — not locally. Requirements:
- GPU: 1× A100 80GB (SXM preferred; PCIe works)
- Template: RunPod PyTorch 2.1+ (CUDA 12.1)
- Setup: run `bash /workspace/schema/scripts/setup_runpod.sh` first

**Training details**:
- Base model: Qwen3-VL-8B-Instruct (quantized 4-bit NF4, ~5GB VRAM)
- LoRA adapters: r=64, alpha=128, applied to attention + MLP layers
- Effective batch size: 16 (2 per device × 8 gradient accumulation steps)
- Max sequence length: 16,384 tokens (covers 1K image + 8K HTML + output safely)
- Duration: ~3-6 hours on A100 80GB for 3K examples
- Cost: ~$8-16 at RunPod Secure rates

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# ── RunPod working directory ────────────────────────────────────────────────
# Screenshot paths in the JSONL are relative (e.g. data/screenshots/foo.png).
# Setting cwd to the project root ensures they resolve correctly.
PROJECT_ROOT = Path('/workspace/schema')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env', override=True)

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DATA_DIR   = PROJECT_ROOT / 'data'          # train.jsonl, eval.jsonl, screenshots/
MODELS_DIR = Path('/workspace/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Working dir: {os.getcwd()}')

## Load Config and Data

In [ ]:
import yaml

with open(PROJECT_ROOT / 'configs/training_config.yaml') as f:
    config = yaml.safe_load(f)

MODEL_ID  = config['model']['name']
HF_TOKEN  = os.getenv('HF_TOKEN')
OUTPUT_DIR = MODELS_DIR / config['training']['output_dir'].split('/')[-1]

print(f'Base model: {MODEL_ID}')
print(f'Output dir: {OUTPUT_DIR}')

# Training data lives in /workspace/schema/data/ (downloaded from HF Hub by setup script)
train_path = DATA_DIR / 'train.jsonl'
eval_path  = DATA_DIR / 'eval.jsonl'
assert train_path.exists(), f'Missing training data at {train_path}\nRun setup_runpod.sh first!'
assert eval_path.exists(),  f'Missing eval data at {eval_path}'

n_train = sum(1 for _ in open(train_path))
n_eval  = sum(1 for _ in open(eval_path))
print(f'Training examples: {n_train}')
print(f'Eval examples:     {n_eval}')

## Load Model with 4-bit Quantization

In [ ]:
from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,   # requires transformers >= 4.53.0
    BitsAndBytesConfig,
)
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f'Loading {MODEL_ID} with 4-bit quantization...')
print('(Downloading ~16GB if not cached, loading to GPU — expect 2-5 min)')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HF_TOKEN,
    cache_dir=str(MODELS_DIR),
)
print('Model weights loaded. Loading processor...')

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    cache_dir=str(MODELS_DIR),
)

print(f'✓ Model ready. Parameters: {model.num_parameters()/1e9:.2f}B')
print(f'  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')
if torch.cuda.is_available():
    print(f'  VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training (handles gradient checkpointing with 4-bit)
model = prepare_model_for_kbit_training(model)

lora_cfg = config['lora']
lora_config = LoraConfig(
    r=lora_cfg['r'],
    lora_alpha=lora_cfg['alpha'],
    lora_dropout=lora_cfg['dropout'],
    bias=lora_cfg['bias'],
    task_type=lora_cfg['task_type'],
    target_modules=lora_cfg['target_modules'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Prepare Dataset

In [ ]:
import json
import torch
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class JSONLDataset(Dataset):
    """Lightweight wrapper — avoids PyArrow schema inference on mixed content types."""
    def __init__(self, path):
        self.data = [json.loads(l) for l in open(path)]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        return self.data[i]

train_dataset = JSONLDataset(train_path)
eval_dataset  = JSONLDataset(eval_path)

print(f'Train: {len(train_dataset)} examples | Eval: {len(eval_dataset)} examples')

# Verify all examples have screenshots
def _has_image(ex):
    for m in ex['messages']:
        if isinstance(m.get('content'), list):
            if any(p.get('type') == 'image' for p in m['content']):
                return True
    return False

n_with_img = sum(1 for i in range(len(train_dataset)) if _has_image(train_dataset[i]))
print(f'Examples with screenshot: {n_with_img}/{len(train_dataset)} train')
assert n_with_img == len(train_dataset), 'Some examples are missing screenshots!'

# Pre-compute assistant token sequence for label masking
_ASST_IDS = processor.tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)

def _find_response_start(ids: list) -> int:
    """Return index of first response token (after <|im_start|>assistant\\n)."""
    n = len(_ASST_IDS)
    for i in range(len(ids) - n, -1, -1):
        if ids[i:i + n] == _ASST_IDS:
            return i + n
    return 0  # fallback: train on whole sequence

def collate_fn(examples):
    """
    On-the-fly collation for Qwen3-VL:
    - Processes images at batch time (avoids PyArrow offset overflow)
    - Masks prompt tokens so we only train on the assistant response
    """
    texts, all_images = [], []

    for ex in examples:
        messages = ex['messages']
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
        img_inputs, _ = process_vision_info(messages)
        all_images.extend(img_inputs or [])

    inputs = processor(
        text=texts,
        images=all_images if all_images else None,
        padding=True,
        truncation=True,
        max_length=config['training']['max_seq_length'],
        return_tensors='pt',
    )

    labels = inputs['input_ids'].clone()
    for i in range(len(examples)):
        ids = labels[i].tolist()
        resp_start = _find_response_start(ids)
        labels[i, :resp_start] = -100          # mask prompt
    labels[inputs['attention_mask'] == 0] = -100  # mask padding
    inputs['labels'] = labels

    return inputs

print('Dataset and collate_fn ready.')

## Training

In [ ]:
from transformers import TrainingArguments, Trainer

train_cfg = config['training']

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=train_cfg['epochs'],
    per_device_train_batch_size=train_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps=train_cfg['gradient_accumulation_steps'],
    learning_rate=train_cfg['learning_rate'],
    lr_scheduler_type=train_cfg['lr_scheduler_type'],
    warmup_ratio=train_cfg['warmup_ratio'],
    bf16=train_cfg['bf16'],
    gradient_checkpointing=train_cfg['gradient_checkpointing'],
    logging_steps=train_cfg['logging_steps'],
    save_steps=train_cfg['save_steps'],
    eval_steps=train_cfg['eval_steps'],
    eval_strategy='steps',
    save_total_limit=train_cfg['save_total_limit'],
    load_best_model_at_end=train_cfg['load_best_model_at_end'],
    metric_for_best_model=train_cfg['metric_for_best_model'],
    report_to=train_cfg.get('report_to', 'none'),
    dataloader_num_workers=0,   # 0 avoids pickling issues with custom collate_fn
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

print('Training configuration:')
print(f'  Effective batch size: {train_cfg["per_device_train_batch_size"] * train_cfg["gradient_accumulation_steps"]}')
print(f'  Learning rate: {train_cfg["learning_rate"]}')
print(f'  Epochs: {train_cfg["epochs"]}')

In [ ]:
# Start training
print('Starting fine-tuning...')
trainer.train()

# Save final model
trainer.save_model(str(OUTPUT_DIR / 'final'))
processor.save_pretrained(str(OUTPUT_DIR / 'final'))

print(f'\nTraining complete. Model saved to {OUTPUT_DIR / "final"}')

In [ ]:
# Training loss curve
import matplotlib.pyplot as plt
import pandas as pd

log_history = trainer.state.log_history
train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs = [l for l in log_history if 'eval_loss' in l]

if train_logs:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot([l['step'] for l in train_logs], [l['loss'] for l in train_logs], label='Train loss')
    if eval_logs:
        ax.plot([l['step'] for l in eval_logs], [l['eval_loss'] for l in eval_logs], label='Eval loss')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Fine-tuning Loss')
    ax.legend()
    plt.tight_layout()
    plt.show()

print('Next step: 08_evaluation.ipynb')